In [ ]:
# file: q4_ocean_dilation.py
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = (128, 128, 1)  # mel x time
num_classes = 3

def dilation_block(x, filters, kernel=(3,3), dilation_rate=1):
    x = layers.Conv2D(filters, kernel, padding='same', dilation_rate=dilation_rate, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    return x

def build_dilation_cnn(input_shape=(128,128,1), num_classes=3):
    inp = layers.Input(shape=input_shape)
    x = dilation_block(inp, 32, dilation_rate=1)
    x = dilation_block(x, 32, dilation_rate=2)
    x = layers.MaxPool2D((2,2))(x)

    x = dilation_block(x, 64, dilation_rate=1)
    x = dilation_block(x, 64, dilation_rate=4)
    x = layers.MaxPool2D((2,2))(x)

    x = dilation_block(x, 96, dilation_rate=1)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(192, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inp, out)
    return model

model = build_dilation_cnn(input_shape, num_classes)
model.summary()
# ensure summary reports params <= 350k

logdir = "logs/ocean/run1"
tb_cb = tf.keras.callbacks.TensorBoard(logdir, histogram_freq=1)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[tb_cb, tf.keras.callbacks.EarlyStopping(patience=8)])